# Diverse Prototypical Ensembles (DPE) — Waterbirds End-to-End Reproduction

This notebook reproduces the **DPE (Diverse Prototypical Ensembles)** baseline on the **Waterbirds** dataset from the ICML 2025 paper:
> **Diverse Prototypical Ensembles Improve Robustness to Subpopulation Shift** (To et al., ICML 2025 / arXiv:2505.23027).

### Reproduction Pipeline:
1. **Verify GPU Hardware**: Confirm NVIDIA GPU availability (recommended: Colab Tesla T4 with 15 GB VRAM).
2. **Install Dependencies**: Install required packages (`netcal`, `jsonargparse`, `transformers`, `seaborn`).
3. **Download & Prepare Waterbirds**: Automatically download the official Stanford NLP archive, unpack images to `data/waterbirds/waterbird_complete95_forest2water2`, and generate the verified `metadata_waterbirds.csv`.
4. **Stage 0 (Feature Backbone Training & Feature Pre-extraction)**: Fine-tune ImageNet-pretrained ResNet-50 for 100 epochs with ERM on Waterbirds images and pre-extract normalized 2048-dim feature representations for validation and test splits (`feats_val.npy`, `feats_test.npy`).
5. **Stage 1 (Diverse Prototypical Ensembles - Original Python Implementation)**:
   - Evaluate the baseline ERM model on the test split.
   - Construct a subgroup-balanced subset from the validation set for prototype learning.
   - Define and train the 15-stage DPE prototype ensemble with IsoMax+ loss and decaying covariance regularization (`cov_reg=5e5`).
   - Run the two paper ablations: **Random subset only** (`cov_reg=0`) and **No diversification** (`cov_reg=0, random_subset=False`).
6. **Visualizations & Per-Group Analysis**:
   - Plot Worst-Group Accuracy (WGA) and Balanced Accuracy curves vs. number of prototypes.
   - Plot per-group test accuracy bar chart across all 4 subgroups (Landbird on Land, Landbird on Water, Waterbird on Land, Waterbird on Water).
7. **Paper Benchmark Comparison**: Compare the reproduced metrics with published numbers in the paper.

## 1. Verify GPU Hardware
Ensure that a GPU is available and check device memory.

In [ ]:
# Detect Google Colab environment and ensure working directory is repo root
import os
import sys

if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    if not os.path.exists('scripts/prepare_waterbirds.py'):
        if not os.path.exists('waterbirds-shortcut-mitigation'):
            !git clone https://github.com/chautanphat/waterbirds-shortcut-mitigation.git
        %cd waterbirds-shortcut-mitigation

print(f'Current working directory: {os.getcwd()}')
!nvidia-smi


## 2. Install Dependencies & Setup Environment
Install necessary third-party libraries for calibration evaluation, argument parsing, and visualization.

In [ ]:
!pip install -q netcal jsonargparse transformers seaborn

## 3. Download & Format Waterbirds Dataset
Downloads `waterbird_complete95_forest2water2.tar.gz` from Stanford NLP, extracts images to `data/waterbirds/waterbird_complete95_forest2water2`, formats `metadata_waterbirds.csv`, and runs group verification.

In [ ]:
!python scripts/prepare_waterbirds.py --data_dir data

## 4. Stage 0: ResNet-50 Feature Backbone Training & Feature Pre-extraction
Fine-tunes an ImageNet-pretrained ResNet-50 for 100 epochs with ERM on Waterbirds images and pre-extracts normalized 2048-dim feature representations for validation and test splits (`feats_val.npy`, `feats_test.npy`).

*Estimated runtime on Colab T4:* ~25–35 minutes.

In [ ]:
!mkdir -p checkpoints/waterbirds_stage0
!python baselines/dpe4subpop/main.py \
  --dataset_name Waterbirds \
  --data_dir data \
  --ckpt_dir checkpoints/waterbirds_stage0 \
  --model_name resnet50 \
  --pretrained_imgnet \
  --train_mode full \
  --loss_name ce \
  --epochs 100 \
  --lr 0.001 \
  --momentum 0.9 \
  --optim sgd \
  --batch_size 128 \
  --batch_size_eval 256 \
  --workers 4 \
  --force_saving_feats \
  --no_wandb \
  --seed 0


## 5. Stage 1: Diverse Prototypical Ensemble (DPE) Training (Original Python Implementation)
This section directly executes the original Python implementation from `baselines/dpe4subpop/notebooks/01_waterbirds_with_attribute_annotation.ipynb`.
It loads the pre-extracted features from Stage 0, evaluates the ERM baseline, trains the 15-stage DPE prototype ensemble, and runs the diversification ablations (`Random subset` and `No diversification`).

*Estimated runtime on Colab T4:* ~1–2 minutes.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from functools import partial
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure baseline modules are in sys.path
repo_root = os.path.abspath('.')
dpe_root = os.path.join(repo_root, 'baselines', 'dpe4subpop')
if dpe_root not in sys.path:
    sys.path.insert(0, dpe_root)

from utils.misc import fix_random_seed
from utils import datasets as dsets
from utils.eval_helpers import eval_metrics, get_acc
from utils.isomaxplus import IsoMaxPlusLossFirstPart, IsoMaxPlusLossSecondPart
from notebooks.src.utils_plot import (
    dict_to_df,
    plot_metrics,
    show_erm_per_group_accuracy,
    plot_distributions,
    show_examples
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
fix_random_seed(0)

In [ ]:
dataset_name = 'Waterbirds'
data_dir = 'data'
ckpt_dir = 'checkpoints/waterbirds_stage0'
workers = 2 if os.name == 'nt' else 4
batch_size_train, batch_size_eval = 256, 256

groups_dict = {
    '00': 'Landbird on Land',
    '01': 'Landbird on Water',
    '10': 'Waterbird on Land',
    '11': 'Waterbird on Water'
}

datasets, dataloaders = dict(), dict()
datasets['val'] = vars(dsets)[dataset_name](data_dir, 'va', None)
datasets['test'] = vars(dsets)[dataset_name](data_dir, 'te', None)
datasets['train_eval'] = vars(dsets)[dataset_name](data_dir, 'tr', None)

def get_pre_extracted_features(_ckpt_dir: str, _set_name: str) -> np.ndarray:
    pre_extracted_feats = np.load(f'{_ckpt_dir}/feats_{_set_name}.npy', mmap_mode='r')
    pre_extracted_feats = ((pre_extracted_feats - pre_extracted_feats.mean(axis=1, keepdims=True)) /
                           pre_extracted_feats.std(axis=1, keepdims=True))
    return pre_extracted_feats

for set_name in ['val', 'test']:
    datasets[set_name].feats = get_pre_extracted_features(ckpt_dir, set_name)
    dataloaders[set_name] = DataLoader(
        dataset=datasets[set_name],
        num_workers=workers,
        pin_memory=False,
        batch_size=batch_size_eval,
        shuffle=False,
        drop_last=False
    )

print("Pre-extracted features loaded successfully.")
print(f"Val split size: {len(datasets['val'])}, Test split size: {len(datasets['test'])}")

In [ ]:
def init_erm_model(_ckpt_path, num_classes=2, device='cuda'):
    ckpt = torch.load(_ckpt_path, map_location="cpu")
    _model = torchvision.models.resnet50()
    backbone = torch.nn.Sequential(*list(_model.children())[:-1])
    emb_dim = _model.fc.in_features
    head = nn.Linear(emb_dim, num_classes)

    _model = nn.Sequential(backbone, nn.Flatten(), head)
    _model.load_state_dict(ckpt, strict=True)
    _model.to(device)
    return _model

def evaluate(_model, eval_loader, device='cuda'):
    ds = eval_loader.dataset
    classes, attributes, groups = np.array(ds.y), np.array(ds._a), np.array(ds.g)
    _model.eval()
    all_preds = []
    with torch.no_grad():
        for *_, feats in eval_loader:
            feats = feats.to(device)
            outputs = _model[-1](feats)
            all_preds.append(outputs.detach().softmax(1).cpu())
        all_preds = torch.concat(all_preds, dim=0).numpy()
        _res = eval_metrics(all_preds, np.array(classes), np.array(attributes), np.array(groups))
    return _res

erm_model = init_erm_model(f'{ckpt_dir}/ckpt_last.pt', device=device)
results = dict()
results['ERM'] = evaluate(erm_model, dataloaders['test'], device=device)

print(f"ERM Overall Accuracy: {results['ERM']['overall']['accuracy'] * 100:.2f}%")
print(f"ERM Worst-Group Accuracy (WGA): {results['ERM']['min_group']['accuracy'] * 100:.2f}%")
show_erm_per_group_accuracy(results['ERM'], groups_dict)
plt.show()

In [ ]:
def get_subsampled_train_set(_datasets=None, train_attr='yes', subsample_type='group', _dataset_name='Waterbirds', trn_split='va'):
    if _datasets is None:
        pre_extracted_feats = None
        _datasets = {}
    else:
        pre_extracted_feats = _datasets['val'].feats
    _datasets['train'] = vars(dsets)[_dataset_name](
        data_dir, trn_split, None, train_attr=train_attr, subsample_type=subsample_type,
        pre_extracted_feats=pre_extracted_feats
    )
    return _datasets

def get_train_loader(_datasets, attr_availability='yes'):
    _datasets = get_subsampled_train_set(_datasets, attr_availability)
    train_loader = DataLoader(
        _datasets['train'],
        batch_size=batch_size_train,
        drop_last=True,
        shuffle=True,
        num_workers=workers,
        pin_memory=False
    )
    return train_loader

plot_distributions(
    get_subsampled_train_set(datasets),
    groups_dict,
    fig_size=(12, 3),
    title='Training Set Subsampled from Validation Set (Subgroup-Balanced)',
    set_name_dict={'train': ''}
)
plt.show()

In [ ]:
def init_model(_ckpt_path, num_classes=2, _model=None, device='cuda'):
    """The backbone is frozen so we only load it once"""
    if _model is None:
        ckpt = torch.load(_ckpt_path, map_location="cpu")
        _model = torchvision.models.resnet50()
        backbone = torch.nn.Sequential(*list(_model.children())[:-1])
        emb_dim = _model.fc.in_features
        head = IsoMaxPlusLossFirstPart(emb_dim, num_classes)

        _model = nn.Sequential(backbone, nn.Flatten(), head)
        _model.load_state_dict(ckpt, strict=False)
        _model.emb_dim = emb_dim
    else:
        assert hasattr(_model, "emb_dim")
        _model[-1] = IsoMaxPlusLossFirstPart(_model.emb_dim, num_classes)
    _model.to(device)
    return _model

def train_prototypes(train_loader, val_loader, model, _prototype_ensemble=(),
                     epochs=20, cov_reg=5e5, wd_weight=10, device='cuda',
                     entropic=30, lr=1e-3, _stage=1, verbose=True):
    best_val_wga, val_wga = 0.0, 0.0
    best_val_wga_prototype = None

    criterion = IsoMaxPlusLossSecondPart(entropic_scale=entropic, reduction='mean')
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    if len(_prototype_ensemble) > 0:
        _prototype_ensemble = torch.concat([_[0] for _ in _prototype_ensemble], dim=1).detach()

    for epoch in range(epochs):
        model.train()
        running_loss, running_clf, running_cov, running_correct, total = 0.0, 0.0, 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"[Stage {_stage}] Epoch {epoch + 1}",
                    leave=(epoch == epochs - 1) and verbose)

        for _, _, labels, _, _, feats in pbar:
            feats = feats.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model[-1](feats)
            clf_loss = criterion(outputs, labels)

            head = model[-1]
            wd = torch.einsum('ijk,ilk->ijl', [head.prototypes[:, None], head.prototypes[:, None]]) * wd_weight
            wd = wd.squeeze().mean()
            loss = clf_loss + wd

            cov_loss = torch.tensor(0.0, device=device)
            if len(_prototype_ensemble) and (cov_reg > 0):
                _prototypes = torch.cat([head.prototypes[:, None], _prototype_ensemble], dim=1)
                n_pro, n_dim = _prototypes.shape[1:]
                cov = torch.einsum('ijk,ilk->ijl', [_prototypes, _prototypes]) / (n_dim - 1)
                cov_loss = torch.abs(cov[:, 0, 1:].sum(1).div(n_pro).mean())
                loss += cov_loss * cov_reg

            loss.backward()
            optimizer.step()

            preds = outputs.argmax(dim=1)
            correct = (preds == labels).sum().item()
            running_loss += loss.item()
            running_clf += clf_loss.item()
            running_cov += cov_loss.item()
            running_correct += correct
            total += labels.size(0)

            pbar.set_postfix({
                'loss': running_loss / (total // labels.size(0)),
                'clf': running_clf / (total // labels.size(0)),
                'cov': running_cov / (total // labels.size(0)),
                'acc': f"{running_correct / total:.2%}",
                'val wga': f"{val_wga:.2%}"
            })

        val_wga = evaluate(model, val_loader, device=device)['min_group']['accuracy']
        if val_wga >= best_val_wga:
            best_val_wga = val_wga
            best_val_wga_prototype = [
                model[-1].prototypes[:, None].detach().clone(),
                model[-1].distance_scale.detach().clone()
            ]

    return best_val_wga_prototype

def cov_reg_scheduler_inv(_cov_reg, _num_stages, alpha=0.05):
    scheduler = [_cov_reg / (1 + alpha * _) for _ in range(_num_stages)]
    scheduler = [_cov_reg] + scheduler

    def cov_reg_schedule(_stage):
        _stage = max(_stage, 1)
        return scheduler[_stage - 1]

    return cov_reg_schedule

def evaluate_ensemble(_prototype_ensemble, eval_loader, _model, device='cuda', show_individuals=False, verbose=True):
    dist_scales = [_[1].detach() for _ in _prototype_ensemble]
    clf = torch.concat([_[0] for _ in _prototype_ensemble], dim=1).detach().transpose(0, 1)
    preds_list = torch.zeros(clf.shape[0], len(eval_loader.dataset), eval_loader.dataset.num_labels)

    ds = eval_loader.dataset
    classes, attributes, groups = np.array(ds.y), np.array(ds._a), np.array(ds.g)

    position = 0

    with torch.no_grad():
        for *_, feats in tqdm(eval_loader, leave=False):
            feats = feats.to(device)

            for i, _weight in enumerate(clf):
                _model[-1].prototypes = torch.nn.Parameter(_weight, requires_grad=False)
                _model[-1].distance_scale = nn.Parameter(dist_scales[i], requires_grad=False)
                _model.eval()
                preds_list[i][position:position + feats.shape[0]] = _model[-1](feats.squeeze())
            position += feats.shape[0]

    if show_individuals:
        for i in range(preds_list.shape[0] - 1, -1, -1):
            preds = preds_list[i].softmax(1).argmax(1).numpy()
            get_acc(preds, classes, groups)

    get_acc(preds_list.softmax(2).mean(0).argmax(1).numpy(), classes, groups, verbose=verbose)

    preds = preds_list.softmax(2).mean(0).detach().cpu().numpy()
    _res = eval_metrics(preds, classes, attributes, groups)
    if verbose:
        print(f"Ensemble WGA: {_res['min_group']['accuracy'] * 100:.1f}%")
    return _res

def train_ensemble(init_model_func, _datasets, _dataloaders, cov_reg=5e5, random_subset=True, num_stages=15, show_freq=15, seed=0,
                   epochs=20, attr_availability='yes', device='cuda'):
    fix_random_seed(seed)
    full_model = None
    prototype_ensemble = []
    ensemble_wga, ensemble_acc, ensemble_balanced_acc = [], [], []
    _train_prototypes = partial(train_prototypes, val_loader=_dataloaders['val'],
                                epochs=epochs, wd_weight=10, device=device, entropic=30, lr=1e-3)
    cv_scheduler = cov_reg_scheduler_inv(cov_reg, num_stages)
    train_loader = get_train_loader(_datasets, attr_availability=attr_availability)

    for stage in range(1, num_stages + 1):
        verbose = stage % show_freq == 0 or stage == 1 or stage == num_stages + 1
        full_model = init_model_func(_model=full_model)
        prototype_ensemble.append(_train_prototypes(
            train_loader, model=full_model,
            _prototype_ensemble=prototype_ensemble, _stage=stage, verbose=verbose,
            cov_reg=cv_scheduler(stage),
        ))
        res = evaluate_ensemble(prototype_ensemble, _dataloaders['test'], full_model, device=device, verbose=verbose)
        ensemble_wga.append(res['min_group']['accuracy'])
        ensemble_acc.append(res['overall']['accuracy'])
        ensemble_balanced_acc.append(res['overall']['balanced_acc'])

        if stage <= num_stages + 1 and random_subset:
            train_loader = get_train_loader(_datasets, attr_availability)
    return ensemble_wga, ensemble_balanced_acc, res

In [ ]:
train_ensemble_func = partial(
    train_ensemble,
    _datasets=datasets,
    _dataloaders=dataloaders,
    init_model_func=partial(init_model, _ckpt_path=f'{ckpt_dir}/ckpt_last.pt', device=device),
    num_stages=15,
    epochs=20,
    device=device
)

metrics = dict()

print("=" * 60)
print("1. Training DPE (Full Diversification: Covariance Reg + Random Subsets)")
print("=" * 60)
metrics['DPE'] = train_ensemble_func(cov_reg=5e5, random_subset=True)

print("\n" + "=" * 60)
print("2. Ablation: DPE without Covariance Reg (Random Subsets Only)")
print("=" * 60)
metrics['Random subset'] = train_ensemble_func(cov_reg=0, random_subset=True)

print("\n" + "=" * 60)
print("3. Ablation: DPE without Diversification (Single Fixed Subset, cov_reg=0)")
print("=" * 60)
metrics['No diversification'] = train_ensemble_func(cov_reg=0, random_subset=False)

## 6. Visualizations & Per-Group Analysis
Compare the ensemble performance curves across stages and the final per-group accuracy across subgroups for ERM vs. DPE variants.

In [ ]:
df = dict_to_df(metrics)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
plot_metrics(df, 'Worst Group Accuracy', axes[0])
plot_metrics(df, 'Balanced Accuracy', axes[1], show_legend=True)
plt.show()

In [ ]:
results = dict()
results['ERM'] = evaluate(erm_model, dataloaders['test'], device=device)
for k in metrics.keys():
    results[k] = metrics[k][-1]

df2 = []
for k in results.keys():
    per_group_acc = results[k]['per_group']
    tmp = {g: per_group_acc[g]['accuracy'] * 100 for g in per_group_acc.keys()}
    df2.append(pd.DataFrame(tmp.values(), columns=['Accuracy']))
    df2[-1]['Group'] = [groups_dict[kk] for kk in tmp.keys()]
    df2[-1]['Diversification Strategy'] = k
df2 = pd.concat(df2)
df2.reset_index(drop=True, inplace=True)

plt.figure(figsize=(16, 4), dpi=200)
ax = sns.barplot(
    data=df2,
    x='Diversification Strategy',
    y='Accuracy',
    hue='Group',
    palette=sns.color_palette('Blues', len(df2['Group'].unique())),
)
ax.set_ylim(70, 101)
ax.set_xlabel('')
ax.set_ylabel('Accuracy (%)')
ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7, axis='y')

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=len(df2['Group'].unique()),
    frameon=True
)
_ = plt.title('Per-group Accuracy of DPE Variants and ERM on Waterbirds', weight='bold')
plt.show()

## 7. Compare with Paper Reported Results

Benchmark your reproduced metrics against the numbers reported in Table 1 & Table 2 of To et al. (ICML 2025):

| Method | Group Info (Train / Val) | Worst-Group Accuracy | Average Accuracy |
| :--- | :---: | :---: | :---: |
| **Standard ERM** | None / None | 69.1% ± 1.8% | 97.2% ± 0.2% |
| **ERM\*** (Backbone Stage 0) | None / None | 77.9% ± 3.0% | 96.9% ± 0.3% |
| **GroupDRO** | Train & Val | 89.9% ± 0.9% | 93.5% ± 0.3% |
| **DFR** | Val only | 92.9% ± 0.4% | 94.0% ± 0.3% |
| **DPE (Random subset)** | Val only | ~92.2% | ~96.5% |
| **DPE (Full - Stage 1+)** | **Val only** | **94.1% ± 0.2%** | **96.8% ± 0.2%** |

*Note on Group Annotations:* In this reproduction notebook (`01_waterbirds_with_attribute_annotation`), subgroup labels on the validation set are utilized for group-balanced subsampling and model selection (as in DFR). The unsupervised version without any subgroup annotations uses class-balanced subsampling (`subsample_type='class'`) as detailed in the paper.